In [16]:
!pip install fastapi uvicorn pyngrok


In [17]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, EmailStr
from typing import Optional, List
import sqlite3
import uvicorn

app = FastAPI(title="Student Database CRUD API")

# Database Initialization
def init_db():
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            student_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            dob TEXT NOT NULL,
            email TEXT,
            phone TEXT,
            course TEXT,
            address TEXT,
            enrollment_date TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

# Pydantic Models for Validation
class StudentCreate(BaseModel):
    name: str
    dob: str
    email: Optional[str] = None
    phone: Optional[str] = None
    course: Optional[str] = None
    address: Optional[str] = None
    enrollment_date: Optional[str] = None

class StudentUpdate(BaseModel):
    name: Optional[str] = None
    dob: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    course: Optional[str] = None
    address: Optional[str] = None
    enrollment_date: Optional[str] = None

# 1. Create a Student
@app.post("/students", status_code=201)
def create_student(student: StudentCreate):
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO students (name, dob, email, phone, course, address, enrollment_date)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (student.name, student.dob, student.email, student.phone, student.course, student.address, student.enrollment_date))
    conn.commit()
    student_id = cursor.lastrowid
    conn.close()
    return {"message": "Student created successfully", "student_id": student_id, **student.dict()}

# 2. List / Filter Students
@app.get("/students")
def get_students(course: Optional[str] = None):
    conn = sqlite3.connect("students.db")
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    if course:
        cursor.execute("SELECT * FROM students WHERE course = ?", (course,))
    else:
        cursor.execute("SELECT * FROM students")

    rows = cursor.fetchall()
    conn.close()
    return [dict(row) for row in rows]

# 3. View a Single Student
@app.get("/students/{student_id}")
def get_student(student_id: int):
    conn = sqlite3.connect("students.db")
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
    row = cursor.fetchone()
    conn.close()

    if not row:
        raise HTTPException(status_code=404, detail="Student not found")
    return dict(row)

# 4. Update Student Details
@app.put("/students/{student_id}")
def update_student(student_id: int, student: StudentUpdate):
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
    if not cursor.fetchone():
        conn.close()
        raise HTTPException(status_code=404, detail="Student not found")

    update_data = {k: v for k, v in student.dict().items() if v is not None}
    if not update_data:
        conn.close()
        return {"message": "No updates provided"}

    set_clause = ", ".join([f"{k} = ?" for k in update_data.keys()])
    values = list(update_data.values()) + [student_id]

    cursor.execute(f"UPDATE students SET {set_clause} WHERE student_id = ?", values)
    conn.commit()
    conn.close()
    return {"message": "Student updated successfully"}

# 5. Delete a Student
@app.delete("/students/{student_id}")
def delete_student(student_id: int):
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
    if not cursor.fetchone():
        conn.close()
        raise HTTPException(status_code=404, detail="Student not found")

    cursor.execute("DELETE FROM students WHERE student_id = ?", (student_id,))
    conn.commit()
    conn.close()
    return {"message": "Student deleted successfully"}

Overwriting main.py


In [18]:
import subprocess
import time
from pyngrok import ngrok

# Set your ngrok authtoken
NGROK_TOKEN = "3KwzjrMnIpLEbuimsKbLkQr3Ohc_3UnCFyQStuQxmHjUHAhwS"
ngrok.set_auth_token(NGROK_TOKEN)

# Start FastAPI server in the background
subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(2)  # Allow server to start

# Open a ngrok tunnel to port 8000
public_url = ngrok.connect(8000)
print(f"Swagger UI Documentation available at: {public_url}/docs")


Swagger UI Documentation available at: NgrokTunnel: "https://runway-shortwave-maybe.ngrok-free.dev" -> "http://localhost:8000"/docs


In [19]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import sqlite3

app = FastAPI(title="Student Database CRUD API")

# Database Initialization
def init_db():
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            student_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            dob TEXT NOT NULL,
            email TEXT,
            phone TEXT,
            course TEXT,
            address TEXT,
            enrollment_date TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

# Pydantic Models for Validation
class StudentCreate(BaseModel):
    name: str
    dob: str
    email: Optional[str] = None
    phone: Optional[str] = None
    course: Optional[str] = None
    address: Optional[str] = None
    enrollment_date: Optional[str] = None

class StudentUpdate(BaseModel):
    name: Optional[str] = None
    dob: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    course: Optional[str] = None
    address: Optional[str] = None
    enrollment_date: Optional[str] = None

# 1. Create a Student
@app.post("/students", status_code=201)
def create_student(student: StudentCreate):
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO students (name, dob, email, phone, course, address, enrollment_date)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (student.name, student.dob, student.email, student.phone, student.course, student.address, student.enrollment_date))
    conn.commit()
    student_id = cursor.lastrowid
    conn.close()
    return {"message": "Student created successfully", "student_id": student_id, **student.dict()}

# 2. List / Filter Students
@app.get("/students")
def get_students(course: Optional[str] = None):
    conn = sqlite3.connect("students.db")
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    if course:
        cursor.execute("SELECT * FROM students WHERE course = ?", (course,))
    else:
        cursor.execute("SELECT * FROM students")

    rows = cursor.fetchall()
    conn.close()
    return [dict(row) for row in rows]

# 3. View a Single Student
@app.get("/students/{student_id}")
def get_student(student_id: int):
    conn = sqlite3.connect("students.db")
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
    row = cursor.fetchone()
    conn.close()

    if not row:
        raise HTTPException(status_code=404, detail="Student not found")
    return dict(row)

# 4. Update Student Details
@app.put("/students/{student_id}")
def update_student(student_id: int, student: StudentUpdate):
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
    if not cursor.fetchone():
        conn.close()
        raise HTTPException(status_code=404, detail="Student not found")

    update_data = {k: v for k, v in student.dict().items() if v is not None}
    if not update_data:
        conn.close()
        return {"message": "No updates provided"}

    set_clause = ", ".join([f"{k} = ?" for k in update_data.keys()])
    values = list(update_data.values()) + [student_id]

    cursor.execute(f"UPDATE students SET {set_clause} WHERE student_id = ?", values)
    conn.commit()
    conn.close()
    return {"message": "Student updated successfully"}

# 5. Delete a Student
@app.delete("/students/{student_id}")
def delete_student(student_id: int):
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
    if not cursor.fetchone():
        conn.close()
        raise HTTPException(status_code=404, detail="Student not found")

    cursor.execute("DELETE FROM students WHERE student_id = ?", (student_id,))
    conn.commit()
    conn.close()
    return {"message": "Student deleted successfully"}

Overwriting main.py


In [20]:
from pyngrok import ngrok
import subprocess
import time

# Set your ngrok authtoken (replace with your token from ngrok dashboard)
!ngrok config add-authtoken 3KwzjrMnIpLEbuimsKbLkQr3Ohc_3UnCFyQStuQxmHjUHAhwS

# Start FastAPI server in the background
subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(2)

# Open a ngrok tunnel to port 8000
public_url = ngrok.connect(8000)
print(f"Swagger UI Documentation available at: {public_url}/docs")


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Swagger UI Documentation available at: NgrokTunnel: "https://runway-shortwave-maybe.ngrok-free.dev" -> "http://localhost:8000"/docs


In [21]:
from pyngrok import ngrok
import subprocess
import time

# Disconnect and kill any existing ngrok processes/tunnels to free up endpoints
try:
    ngrok.kill()
except Exception:
    pass

# Set your ngrok authtoken
!ngrok config add-authtoken 3KwzjrMnIpLEbuimsKbLkQr3Ohc_3UnCFyQStuQxmHjUHAhwS

# Start FastAPI server in the background
subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(2)

# Open a ngrok tunnel to port 8000
public_url = ngrok.connect(8000)
print(f"Swagger UI Documentation available at: {public_url}/docs")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Swagger UI Documentation available at: NgrokTunnel: "https://runway-shortwave-maybe.ngrok-free.dev" -> "http://localhost:8000"/docs


In [22]:
!pip freeze > requirements.txt